# 01 · Define & Explore — spec the filter API + implement Layer 1

**Standard slot:** *define & explore.* **For Project 05 this means:** there is no target to explore —
you explore the **API you are building**. Walk through the `Design` dataclass and the four layer
functions in the shared engine, internalize the **discrimination problem**, then implement and verify
**Layer 1 (self-consistency)** with inline asserts on planted known-good / known-bad `EXAMPLE_DATA`
designs (D0).

Run `00_setup.ipynb` first in this session.

## The discrimination problem (read this first)

A campaign produces **thousands** of candidate designs, each with in-silico metrics. Triage means
returning a small, ranked, defensible short-list. The uncomfortable truth this whole project is built
around:

> **No in-silico metric — and no single layer — perfectly separates true hits from false ones.
> Filters _enrich_ the pool; they do not _guarantee_ a hit.**

Each layer raises the fraction of truly-good designs among the survivors (good), but also discards
some real hits (false negatives) and lets some duds through (false positives). Your deliverable is a
clean, **tested** engine *and* an honest map of where it fails — not a perfect classifier.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## The `Design` dataclass — the unit the engine scores

The shared engine operates on `fp.Design` records: one designed protein (monomer/binder/enzyme/
antibody/oligomer) plus whatever metrics your upstream predictors produced. The student's job in a
real campaign (Projects 01/03) is to *populate* these fields; this engine *scores* them. Let's read
the contract.

In [ ]:
import filtering_pipeline as fp
import inspect

# The data model the whole engine revolves around:
print(inspect.getsource(fp.Design))

### Key fields, by which layer consumes them

| Field | Layer | Meaning |
|-------|-------|---------|
| `scrmsd`, `plddt`, `pae_interaction` | L1 self-consistency | does the sequence fold back to the design? (pLDDT is confidence, **NOT** stability) |
| `plddt_catalytic`, `catalytic_geom_rmsd` | L1 (enzymes) | active-site confidence + theozyme geometry |
| `scrmsd_orthogonal` | L2 orthogonal | a *second* predictor (ESMFold/Boltz) agrees |
| `solubility`, `rosetta_dG`, `shape_complementarity` | L3 physics | aggregation risk + (binders) interface energy & packing |
| `md_rmsd` | L4 dynamics | structure does not drift over a short MD |
| `layers_passed`, `score`, `notes` | filled by the pipeline | survival depth, composite score, audit trail |

## The layer-function signatures — the API you are building

These are the functions you implement and test. Read their signatures and contracts now; you will
verify Layer 1 below, Layers 2+3 in notebook 02, run the whole thing in 03, and analyze it in 04.

In [ ]:
for fn in (fp.self_consistency, fp.orthogonal_check, fp.physics_filter,
           fp.dynamics_filter, fp.rank_designs, fp.run_pipeline, fp.report):
    sig = inspect.signature(fn)
    doc = (fn.__doc__ or "").strip().splitlines()[0]
    print(f"{fn.__name__}{sig}\n    {doc}\n")

## Cutoffs by design type — and why they differ

`DEFAULT_CUTOFFS` holds per-type thresholds. They differ because the design problems differ: an
antibody CDR loop is harder to predict than an idealized helical monomer, so its scRMSD/pLDDT bars
are looser; a binder adds interface terms (`pae`, `rosetta_dG`, `sc`) a monomer does not have.
**Justifying these is a graded task** — cite the self-consistency literature (Dauparas 2022) and your
own enrichment analysis (notebook 04).

In [ ]:
print("DEFAULT_CUTOFFS (starting points from the validation-ref lineage; justify + PR changes):")
for k, v in fp.DEFAULT_CUTOFFS.items():
    print(f"  {k:9s} {v}")

## The planted `EXAMPLE_DATA` fixtures — your "controls"

This project's controls are **planted designs with known answers**: a known-good design that should
pass every layer, and known-bad designs each built to fail a *specific* layer. They live in
`scripts/make_example_pool.py` and drive both the pool and the unit tests, so the two stay in sync.
**These are synthetic — never present their numbers as real.**

In [ ]:
from make_example_pool import known_designs

planted = {r["design_id"]: r for r in known_designs()}
print(f"{len(planted)} planted fixtures:")
for did, row in planted.items():
    print(f"  {did:34s} truth={row['truth']:4s} {row['note']}")

## Implement & verify **Layer 1 — self-consistency**

Layer 1 asks: *does the sequence fold back to the shape it was designed for?* It checks `scrmsd`,
`plddt`, (and for enzymes `plddt_catalytic` + `catalytic_geom_rmsd`, for complexes `pae`) against the
type's cutoffs. Below we build `fp.Design` objects from the planted fixtures and assert the expected
pass/fail — this is your D0 evidence.

In [ ]:
def design_from_row(row):
    """Build a fresh fp.Design from a planted EXAMPLE_DATA row (drop non-Design columns)."""
    drop = {"truth", "note", "design_id"}
    fields = {k: v for k, v in row.items() if k not in drop}
    return fp.Design(design_id=row["design_id"], sequence="M", **fields)

cut_mono = fp.DEFAULT_CUTOFFS["monomer"]
cut_enz = fp.DEFAULT_CUTOFFS["enzyme"]

# Known-GOOD monomer must PASS Layer 1:
good = design_from_row(planted["EXAMPLE_DATA_GOOD_monomer"])
assert fp.self_consistency(good, cut_mono) is True, "known-good monomer should pass L1"

# Known-BAD (high scRMSD: confident about the WRONG fold) must FAIL Layer 1:
bad_scrmsd = design_from_row(planted["EXAMPLE_DATA_BAD_L1_scrmsd"])
assert fp.self_consistency(bad_scrmsd, cut_mono) is False, "high-scRMSD design must fail L1"

# Known-BAD (low pLDDT) must FAIL Layer 1:
bad_plddt = design_from_row(planted["EXAMPLE_DATA_BAD_L1_plddt"])
assert fp.self_consistency(bad_plddt, cut_mono) is False, "low-pLDDT design must fail L1"

# Known-BAD enzyme (broken catalytic geometry) must FAIL Layer 1:
bad_enz = design_from_row(planted["EXAMPLE_DATA_BAD_L1_enzyme_geom"])
assert fp.self_consistency(bad_enz, cut_enz) is False, "enzyme with bad active-site geom must fail L1"

print("Layer 1 verified on planted designs:")
print(f"  GOOD_monomer        passed L1? {good.layers_passed >= 1}  (scrmsd={good.scrmsd}, plddt={good.plddt})")
print(f"  BAD_L1_scrmsd       notes: {bad_scrmsd.notes}")
print(f"  BAD_L1_plddt        notes: {bad_plddt.notes}")
print(f"  BAD_L1_enzyme_geom  notes: {bad_enz.notes}")
print("\nAll Layer-1 asserts passed. (Fixtures are EXAMPLE_DATA — synthetic.)")

### Note: the geometry helper (`ca_rmsd`)

If you *don't* pre-compute `scrmsd`, `self_consistency()` will compute it from `designed_pdb` +
`predicted_pdb` via `fp.ca_rmsd()` (Biopython `Superimposer`, teaching-grade Cα-RMSD). In this course
the upstream projects (01/03) usually provide the number directly; the helper is there for when you
have the PDBs. We don't exercise it here (no PDBs in the synthetic pool).

In [ ]:
import inspect
print(inspect.getsource(fp.ca_rmsd))

## D0 checklist
- [ ] One-page API spec: the `Design` fields + each layer function's signature **and contract**.
- [ ] The discrimination problem stated precisely (filters enrich, not guarantee).
- [ ] Layer 1 verified: planted known-good passes; each planted known-bad fails (printout above).
- [ ] `LOG.md` entry: what you ran, seed, outcome.

**Next:** `02_generate.ipynb` — assemble the labeled pool and implement/verify Layers 2 + 3.